In [ ]:
!pip install tsai 
import sys

# Insert it at position 0 to force highest priority
if '/Users/fokia/CRP_SLB' not in sys.path:
    sys.path.insert(0, '/Users/fokia/CRP_SLB')

from geoxts.benchmark_data import *


In [ ]:
import pandas as pd 
!pip install fastparquet
#We first create downsample the gamma rays to per feet.
df_log = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/logs.parquet', engine='fastparquet')
#df_log = df_log.set_index('wellName')
df_log.loc[df_log['GR'] < -1, 'GR' ] = -1
df_loc = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/loc.parquet', engine='fastparquet')
df_loc = df_loc.reset_index()
df_log = df_loc.merge(df_log, how = 'inner', left_on = 'wellName', right_on = 'wellName')
print(df_log.head())

In [ ]:
df_tops = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/tops.parquet', engine='fastparquet')
df_tops.set_index('well_name', inplace = True)
cols = ['NIOBARA','CODELL', 'FORT_HAYS']
df_tops = df_tops[cols]
df_tops.dropna(inplace = True)
df_tops.head() 

In [ ]:
import numpy as np
# Load the training_well array from the file which contains the wells with good signature cluster
well_array = np.load('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/training_well_list.npy', allow_pickle=True)

In [ ]:
from geoxts.benchmark_data import *

In [ ]:
input_variable = ['GR', 'Depth','Latitude','Longitude']
#extract training data
X_train, y_train = extract_dataset_Xy(df_log,df_tops,well_array, input_variable, wsize=201, top_list_bool=1 )
X_train.shape, y_train.shape

In [ ]:
df_valid_log

In [ ]:
#validing datastet for the recall calculation
df_valid_log = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/logs_50.parquet', engine='fastparquet')
df_valid_log.loc[df_valid_log['GR'] < -1, 'GR' ] = -1
df_valid_log.loc[df_valid_log['GR'] > 400, 'GR' ] = 400
#df_loc = df_loc.reset_index()
df_valid_loc = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/loc_50.parquet', engine='fastparquet')
df_valid_loc = df_valid_loc.reset_index()
df_valid_log = df_valid_loc.merge(df_valid_log, how = 'inner', left_on = 'wellName', right_on = 'wellName')
df_valid_tops = pd.read_csv('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/tops_50.csv')
df_valid_tops = df_valid_tops.set_index('wellName')
cols = ['NIOBARA','CODELL', 'FORT_HAYS']
df_valid_tops = df_valid_tops[cols]

In [ ]:
#create validation dataset 
X_valid, y_valid = extract_dataset_Xy(df_valid_log,df_valid_tops,[],input_variable, wsize=201, top_list_bool=0)
X_valid.shape, y_valid.shape

In [ ]:
from tsai.all import *


In [ ]:
X, y, splits = combine_split_data([X_train, X_valid], [y_train, y_valid])
tfms  = [None, [Categorize()]]
dsets = TSDatasets(X, y, tfms=tfms, splits=splits, inplace=True)
#tensor dataloader
dls = TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=25, batch_tfms=[TSNormalize(by_var=True, range = (0,1))], num_workers=0) 
#dls = TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=25, num_workers=0)  

In [ ]:
dsets

In [ ]:
valid_dl = dls.valid
dls.show_batch()

In [ ]:
#Testing datastet for the recall calculation
df_test_log = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/logs_100.parquet', engine='fastparquet')
df_test_log.loc[df_test_log['GR'] < -1, 'GR' ] = -1
df_test_log.loc[df_test_log['GR'] > 400, 'GR' ] = 400
#df_loc = df_loc.reset_index()
df_test_loc = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/loc_100.parquet', engine='fastparquet')
df_test_loc = df_test_loc.reset_index()
df_test_log = df_test_loc.merge(df_test_log, how = 'inner', left_on = 'wellName', right_on = 'wellName')
df_test_tops = pd.read_csv('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/tops_100.csv')
df_test_tops = df_test_tops.set_index('wellName')
cols = ['NIOBARA','CODELL', 'FORT_HAYS']
df_test_tops = df_test_tops[cols]

In [ ]:
batch_size = 64 # Choose what fits your GPU/Memory
dls = TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=batch_size)


In [ ]:
import sys
sys.path.append('/Users/fokia/CRP_SLB/datasets/colorado/Transformer')

# I'm assuming your class inside the file is named PatchTST!
from PatchTST_backbone import PatchTST_backbone 


In [ ]:
model = PatchTST_backbone(
    c_in=X.shape[1],        # Number of input variables/channels
    target_window=1,        # Output dimension
    context_window=X.shape[2], # Your input sequence length
    patch_len=16,           # Whatever your patch length is
    stride=8,               # Whatever your stride is
    n_markers=4             # 4 markers 
)


In [ ]:
%reload_ext autoreload
%autoreload 2 


In [ ]:
df_log

In [ ]:

# We use Learner to link your data to your model
learn = Learner(dls, model, metrics=accuracy) # You might use rmse or mse for regression!

# Start training (10 epochs, max learning rate of 0.001)
learn.fit_one_cycle(10, lr_max=1e-3)

# To view Training/Validation loss curves later:
# learn.recorder.plot_loss()


In [ ]:
learn.recorder.plot_loss()


In [ ]:
learn.show_results()


In [ ]:
# This prints the exact mapping of Geology Names to Integers
print(dls.vocab)


In [ ]:
import numpy as np

# 1. Grab the actual predictions and targets for the entire Validation set
probs, targets = learn.get_preds(ds_idx=1)

# Grab the very first patch index (you can change this to 100, 200, etc., to see other wells!)
idx = 0  

# Find which class the model guessed (argmax picks the highest probability)
# Make it into an array that repeats that guess 200 times so we can plot it!
predicted_class = probs[idx].argmax().item()
predicted_zones = np.full(200, predicted_class) # 200 length array of the same class

# 2. Grab your Gamma Ray log from the inputs!
# X_valid[idx] is the 3D patch. We grab the 1st variable (index 0) which is usually Gamma Ray
gr_log = X_valid[idx][0]
seq_len = len(gr_log)  # Let Python count exactly how long the sequence is

predicted_zones = np.full(seq_len, predicted_class) 
depths = np.arange(seq_len) 


# ----------------- The Plotting Code -----------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

# Left: True Gamma Ray Input Curve
ax1.plot(gr_log, depths, color='steelblue', linewidth=1)
ax1.set_title(f'GR Log | Patch {idx}')
ax1.set_ylabel('Depth Steps')
ax1.invert_yaxis()  # Geological standard: depth goes down!

# Right: Model Output
ax2.plot(predicted_zones, depths, color='steelblue', drawstyle='steps-pre')
ax2.set_title(f'Model Prediction\n(Guessed: Class {predicted_class} | True: {targets[idx].item()})')
ax2.set_xticks([0, 1, 2, 3, 4, 5]) # Adjust to your total number of marker classes

plt.tight_layout()
plt.show()


In [ ]:
X_well[:, 1, 100]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Grab ONE specific Well ID and lock the randomizer
np.random.seed(42)  
well_id = df_valid_tops.index[10]  

# 2. Extract patches & true labels 
X_well, y_well = extract_dataset_Xy(
    df_valid_log, df_valid_tops, [well_id], input_variable, 
    wsize=201, top_list_bool=0
)

# 3. Predict the sequence
probs, targets, _ = learn.get_X_preds(X_well, y_well)
preds = probs.argmax(dim=-1).numpy().flatten()

# 4. Grab the exact physical depths corresponding to our predictions
# (Index '1' is the DEPTH variable in X_well. '100' is the center of the patch)
actual_depths = X_well[:, 1, 100]

# --- Sort the predictions by Depth so the step function draws a clean line! ---
sort_idx = np.argsort(actual_depths)
actual_depths = actual_depths[sort_idx]
preds = preds[sort_idx]

# 5. Grab the FULL, Raw High-Resolution Gamma Ray curve straight from the dataframe!
well_data = df_valid_log[df_valid_log['wellName'] == well_id]
raw_depths = well_data['DEPTH'].values
raw_gr = well_data['GR'].values


# 6. Make the Beautiful GeoTS Paper Plot!
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

# -------- LEFT: Raw High-Res Gamma Ray Log --------
ax1.plot(raw_gr, raw_depths, color='#5c85b1', alpha=0.9, linewidth=1)
ax1.set_title(f'GR Log | {well_id}')
ax1.set_xlabel('GR (API)')
ax1.set_ylabel('Depth (ft)')

# 🌍 GEOLOGICAL MAGIC: This flips the Y-axis so 6000 is at the top, 7000 at bottom!
ax1.invert_yaxis()  

# -------- RIGHT: Predicted Zone Step Function --------
ax2.plot(preds, actual_depths, color='#5c85b1', linewidth=2, drawstyle='steps-pre')
ax2.set_title('Zone step function')
ax2.set_xlabel('Predicted zone')
ax2.set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Grab ONE specific Well ID and lock the randomizer
np.random.seed(42)  
well_id = df_valid_tops.index[0]  

# 2. Extract patches & true labels 
X_well, y_well = extract_dataset_Xy(
    df_valid_log, df_valid_tops, [well_id], input_variable, 
    wsize=201, top_list_bool=0
)

# 3. Predict the sequence
probs, targets, _ = learn.get_X_preds(X_well, y_well)
preds = probs.argmax(dim=-1).numpy().flatten()

# 4. Grab the exact physical depths corresponding to our predictions
# (Index '1' is the DEPTH variable in X_well. '100' is the center of the patch)
actual_depths = X_well[:, 1, 100]

# --- Sort the predictions by Depth so the step function draws a clean line! ---
sort_idx = np.argsort(actual_depths)
actual_depths = actual_depths[sort_idx]
preds = preds[sort_idx]

# 5. Grab the FULL, Raw High-Resolution Gamma Ray curve straight from the dataframe!
well_data = df_valid_log[df_valid_log['wellName'] == well_id]
raw_depths = well_data['DEPTH'].values
raw_gr = well_data['GR'].values


# 6. Make the Beautiful GeoTS Paper Plot!
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

# -------- LEFT: Raw High-Res Gamma Ray Log --------
ax1.plot(raw_gr, raw_depths, color='#5c85b1', alpha=0.9, linewidth=1)
ax1.set_title(f'GR Log | {well_id}')
ax1.set_xlabel('GR (API)')
ax1.set_ylabel('Depth (ft)')

# 🌍 GEOLOGICAL MAGIC: This flips the Y-axis so 6000 is at the top, 7000 at bottom!
ax1.invert_yaxis()  

# -------- RIGHT: Predicted Zone Step Function --------
ax2.plot(preds, actual_depths, color='#5c85b1', linewidth=2, drawstyle='steps-pre')
ax2.set_title('Zone step function')
ax2.set_xlabel('Predicted zone')
ax2.set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

formation_names = ['NIOBARA', 'CODELL', 'FORT_HAYS']
top_names = ['zone_0'] + formation_names

# --- 1. OVERALL ACCURACY & CLASSIFICATION METRICS ---
probs, targets = learn.get_preds(ds_idx=1)
preds = probs.argmax(dim=-1).numpy().flatten()
targets = targets.numpy().flatten()

print(f"Overall depth-sample accuracy: {(preds == targets).mean():.4f}\n")
print("Per-class metrics:")
print(classification_report(targets, preds, target_names=top_names, digits=2))

print("Confusion matrix (rows=true, cols=pred):")
cm = confusion_matrix(targets, preds)
print(pd.DataFrame(cm, index=top_names, columns=top_names))
print("\n" + "="*50 + "\n")

# --- 2. TOP-PICKING MEAN ABSOLUTE ERROR (MAE) ---
mae_errors = {name: [] for name in formation_names}
print("Top-picking mean absolute error:")

for well_id in df_valid_tops.index:
    true_tops = df_valid_tops.loc[well_id] 
    
    X_well, y_well = extract_dataset_Xy(
        df_valid_log, df_valid_tops, [well_id], input_variable, 
        wsize=201, top_list_bool=0
    )
    if len(X_well) == 0: continue
    
    depths = X_well[:, 1, 100] 
    probs, _, _ = learn.get_X_preds(X_well, y_well)
    well_preds = probs.argmax(dim=-1).numpy().flatten()
    
    sort_idx = np.argsort(depths)
    depths = depths[sort_idx]
    well_preds = well_preds[sort_idx]

    #well_preds = np.maximum.accumulate(well_preds[sort_idx])
    
    for zone_idx, formation in enumerate(formation_names, start=1):
        true_depth = true_tops[formation]
        pred_idx = np.where(well_preds == zone_idx)[0]
        
        if len(pred_idx) > 0 and not pd.isna(true_depth):
            predicted_depth = depths[pred_idx[0]]
            mae_errors[formation].append(abs(predicted_depth - true_depth))

for formation in formation_names:
    if len(mae_errors[formation]) > 0:
        print(f"  {formation:<12} {np.mean(mae_errors[formation]):>5.1f} ft")


In [ ]:
print("Accuracy per marker (True Positive Rate):")
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

for name, acc in zip(top_names, per_class_accuracy):
    print(f"  {name:<12} {acc*100:>5.1f}%")


In [ ]:
print("Accuracy per marker (True Positive Rate):")
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

for name, acc in zip(top_names, per_class_accuracy):
    print(f"  {name:<12} {acc*100:>5.1f}%")